# Views: Saving Queries as Virtual Tables

Some queries are useful again and again, like "which loans are currently
out" or "which loans are overdue". Rewriting them every time is tedious. A
**view** is a saved query that behaves like a virtual table: you query the
view by name, and the database runs the underlying query for you. In this
notebook you will learn views, and you will grow your Library with
reservations and member penalties.

## What a view is

A view is created from a SELECT:

```sql
CREATE VIEW view_name AS
SELECT ... ;
```

After that, `SELECT * FROM view_name` runs the saved query. The view stores
no data of its own, it always reflects the current data in the underlying
tables. Let's see one on Chinook first.

In [ ]:
from rel_db_learning import get_project_connection, run_sql, run_ddl, reset_my_project_data

project = get_project_connection()
reset_my_project_data()

On Chinook, here is a query we might reuse: every album with its artist name.
For demonstration we create a view for it.
If the view already exists from a previous run, the CREATE OR REPLACE form
updates it instead of failing.

In [ ]:
from rel_db_learning import get_chinook_connection
chinook = get_chinook_connection()

run_ddl("""
    CREATE OR REPLACE VIEW album_with_artist AS
    SELECT al.album_id, al.title, ar.name AS artist_name
    FROM album al
    JOIN artist ar ON al.artist_id = ar.artist_id
""", chinook)

run_sql("SELECT * FROM album_with_artist ORDER BY title LIMIT 15", chinook)

Querying the view feels exactly like querying a table, but behind it is the
join we wrote once. Now let's bring views to your Library, along with two new
needs: tracking reservations, and penalizing late returns.

## Growing the Library: reservations, returns, and penalties

This stage adds several things at once:

1. A column `is_returned` (boolean) on **loan**, so we can mark a loan as
   returned without relying only on the return date.
2. A column `penalty_days` (integer) on **member**. When a member returns a
   book late, this grows, and it reduces how many days they may borrow next
   time (our loan period is 15 days, so allowed days = 15 minus penalty_days).
3. A new table **reservation**, so members can reserve books.
4. Three **views** for everyday questions: current loans, overdue loans, and
   pending reservations.

As always, we design all of this in pgModeler, then apply it with the Diff
feature.

### Step 1: Open and Save As

1. Open your `stage_3_design.dbm` file in pgModeler (your current design,
   with book.is_available from the previous notebook).
2. Immediately use **File > Save As** and save it as `stage_4_design.dbm` in
   `student_work/my_project/`. You will make all the Stage 4 changes in this
   new file, keeping a clean history per stage.

### Step 2: Add the two new columns

1. Double-click the **loan** table. Add a column:
   - Name: `is_returned`, Type: `boolean`. Optionally set a default of
     `false` in the column's default value field.
2. Double-click the **member** table. Add a column:
   - Name: `penalty_days`, Type: `integer`. Optionally set a default of `0`.

### Step 3: Add the reservation table

1. Use the table tool to add a new table named `reservation`.
2. Add its columns:
   - `reservation_id`, type `integer`, set as primary key.
   - `book_id`, type `integer`.
   - `member_id`, type `integer`.
   - `reservation_date`, type `date`.
   - `is_fulfilled`, type `boolean`.
3. Now define the two foreign keys as **constraints** on the columns you
   just added. Double-click the reservation table, open the **Constraints**
   tab (foreign keys), and add:
   - A foreign key on `book_id` referencing `book(book_id)`.
   - A foreign key on `member_id` referencing `member(member_id)`.
   pgModeler will draw the relationship lines in the diagram automatically
   once the foreign keys exist.

> **Important:** define the foreign keys as constraints on the columns above,
> do NOT use the relationship drawing tool. The relationship tool would
> create extra foreign-key columns (named like `member_id_member`) in
> addition to the ones you added, leaving you with duplicate columns.
> Defining the foreign key as a constraint on your existing column keeps the
> names clean (`book_id`, `member_id`), matching the loan table.

### Step 4: Add the three views

In pgModeler, a view is defined through its **References** tab. The
documentation describes two ways to do this. We will use the simpler one: a
single expression that holds the whole SELECT.

For each view below:

1. Use the view tool to add a new view, and give it the correct name
   (current_loans, overdue_loans, or pending_reservations).
2. Open the view's **References** tab and add a new reference.
3. Set `Ref. type` to **Expression**.
4. Set `Used in` to **View definition**. This tells pgModeler the expression
   is the entire view body.
5. Paste the matching SELECT (from the next cell) into the **Expression**
   field.
6. Open the **Code Preview** tab to confirm pgModeler generates a
   `CREATE VIEW` with your SELECT.
7. (Optional, for a nicer diagram) in the reference, list the referenced
   tables so pgModeler draws relationships from the view to loan, member,
   book, or reservation.

> **Alternative (structured references):** instead of one expression,
> you can build the view from separate references for the SELECT, FROM, and
> WHERE parts, picking tables and columns from dropdowns. This gives pgModeler
> automatic column tracking, but it takes more clicks. For this course the
> single View definition expression is enough.

The three SELECT statements:

**current_loans** (loans that are not yet returned):

```sql
SELECT l.loan_id, m.full_name, b.title, l.loan_date
FROM loan l
JOIN member m ON l.member_id = m.member_id
JOIN book b ON l.book_id = b.book_id
WHERE l.is_returned = false
```

**overdue_loans** (not returned, and past the 15 day loan period):

```sql
SELECT l.loan_id, m.full_name, b.title, l.loan_date
FROM loan l
JOIN member m ON l.member_id = m.member_id
JOIN book b ON l.book_id = b.book_id
WHERE l.is_returned = false
  AND l.loan_date + INTERVAL '15 days' < CURRENT_DATE
```

**pending_reservations** (reservations not yet fulfilled):

```sql
SELECT r.reservation_id, m.full_name, b.title, r.reservation_date
FROM reservation r
JOIN member m ON r.member_id = m.member_id
JOIN book b ON r.book_id = b.book_id
WHERE r.is_fulfilled = false
```

### Step 5: Generate the changes with Diff and apply them

1. Save your `stage_4_design.dbm`.
2. Open pgModeler's **Diff** tool. Set the source to your model
   (stage_4_design) and the target to your live my_project database
   (the connection you set up in notebook 08).
3. Generate the diff. It will contain ALTER TABLE statements for the new
   columns, a CREATE TABLE for reservation (with its foreign keys), and
   CREATE VIEW statements for the three views.
4. Review the generated SQL, then copy all of it.
5. Paste it into the cell below and run it.

In [ ]:
stage_4_sql = """
--- Paste the SQL that Diff generated (ALTER TABLE for the new
--- columns, CREATE TABLE reservation, and the three CREATE VIEW statements).
"""

run_ddl(stage_4_sql, project)

### Step 6: Validate Stage 4

In [ ]:
from rel_db_learning.dbm_validator import validate_exercise
from rel_db_learning.dbm_validator.exercises import PROJECT_STAGE_4_RESERVATIONS_AND_FINES

validate_exercise(
    project_db=get_project_connection(),
    spec=PROJECT_STAGE_4_RESERVATIONS_AND_FINES,
)

## Putting the new schema to work

The new columns start empty. Let's set is_returned from the existing
return_date: a loan is returned if it has a return date.

In [ ]:
run_ddl("UPDATE loan SET is_returned = (return_date IS NOT NULL)", project)
run_sql("SELECT loan_id, loan_date, return_date, is_returned FROM loan ORDER BY loan_id LIMIT 10", project)

Now add a few reservations so the pending_reservations view has something to
show. We reserve some books for some members.

In [ ]:
run_ddl("""
    INSERT INTO reservation (reservation_id, book_id, member_id, reservation_date, is_fulfilled)
    VALUES
        (1, 1, 5, '2026-01-10', false),
        (2, 3, 8, '2026-01-12', false),
        (3, 7, 2, '2026-01-05', true);
""", project)
run_sql("SELECT * FROM reservation", project)

## Querying the views

Now the payoff. Instead of writing the joins each time, we just query the
views by name.

Current loans (not yet returned):

In [ ]:
run_sql("SELECT * FROM current_loans ORDER BY loan_id LIMIT 10", project)

Overdue loans (not returned and past the 15 day period):

In [ ]:
run_sql("SELECT * FROM overdue_loans ORDER BY loan_id LIMIT 10", project)

Pending reservations:

In [ ]:
run_sql("SELECT * FROM pending_reservations ORDER BY reservation_id", project)

## The penalty in action

The penalty_days column lets us reduce a member's borrowing period after a
late return. Suppose member 5 returned a book late: we add 3 penalty days.

In [ ]:
run_ddl("UPDATE member SET penalty_days = 3 WHERE member_id = 5", project)
run_sql("""
    SELECT full_name, penalty_days, 15 - penalty_days AS allowed_loan_days
    FROM member
    WHERE member_id = 5
""", project)

The member can now borrow for 12 days instead of 15. This is our penalty: not
a fine, but a shorter loan period next time.

## Exercise

Reservation 1 has now been fulfilled (the member picked up the book). Write
an UPDATE that sets is_fulfilled to true for the reservation with
reservation_id 1. Then the check confirms which reservations are still
pending.

In [ ]:
# Write your UPDATE here, then run this cell.
my_update = "???"
run_ddl(my_update, project)

In [ ]:
from rel_db_learning.widgets.self_assessment import sql_query_check

sql_query_check(
    student_sql="SELECT reservation_id FROM pending_reservations ORDER BY reservation_id",
    correct_sql="m04_nb_10_exercise_1",
    connection=project,
)

After your update, only reservation 2 should remain pending (reservation 1 is
now fulfilled, and reservation 3 was already fulfilled). If you want to reset
your data, run `reset_my_project_data()` (note: this restores core data; the
reservation table and views remain, since they are schema, not core seed
data).

## A note on materialized views

The views you created are **ordinary views**: they store no data, and rerun
their query every time you read them. They always reflect the latest data,
but on very large tables the repeated work can be slow.

PostgreSQL also offers **materialized views**, which store the query result
physically, like a cached table:

```sql
CREATE MATERIALIZED VIEW view_name AS
SELECT ... ;
```

A materialized view is fast to read because the result is already computed,
but it does not update automatically. You refresh it on demand:

```sql
REFRESH MATERIALIZED VIEW view_name;
```

The trade-off: ordinary views are always current but recomputed each time;
materialized views are fast but can be stale until refreshed. Use ordinary
views for live, frequently changing data, and materialized views for
expensive queries over data that changes less often (common in reporting and
analytics). In pgModeler, you choose between them with the view's **Mode**
setting (Ordinary or Materialized).

For your Library the data is small and always changing, so ordinary views
are the right choice here.

### Try it yourself: ordinary vs materialized views

The note above described the trade-off. Now see it in action. Below are a base loan table and two views of "loans per member": an ordinary view and a materialized one.

Try this:

- Press "Add loan" for any member. The ordinary view updates at once, because a view re-runs its query every time you read it.
- Notice the materialized view does not change. It still shows its last saved result, so it is now stale (marked in amber).
- Press "REFRESH" to recompute the materialized view. It matches the ordinary view again.

This is the staleness trade-off: an ordinary view is always current but computes every time, while a materialized view is fast to read but only as fresh as its last refresh.

In [ ]:
from rel_db_learning.widgets import view_staleness

view_staleness()

## What's next

Your Library now has views, reservations, and a penalty system. Next you
will learn about **indexes**, which speed up queries on large tables, using
the much larger Chinook database to see the difference.